# Surf Pop-Up Analyzer — Fase 0

**Entregable 1: verificar extracción de pose.** Corre `src/pose.py` sobre un video de `data/raw/` y dibuja el esqueleto sobre el primer frame con pose válida.

Si el esqueleto sigue al cuerpo → la extracción funciona y se justifica seguir con `phases.py` / `metrics.py`.

In [ ]:
import os, glob
import cv2
import numpy as np
import matplotlib.pyplot as plt

from src import pose

# 1) Modelo (descarga heavy.task si falta)
pose.download_model()

# 2) Listar videos disponibles
videos = sorted(
    glob.glob('data/raw/*.mp4') + glob.glob('data/raw/*.mov') +
    glob.glob('data/raw/*.MP4') + glob.glob('data/raw/*.MOV')
)
print('Videos en data/raw/:', videos)
assert videos, 'Dropea al menos un video en data/raw/ y re-corre esta celda.'

In [ ]:
# 3) Extraer landmarks del primer video
video_path = videos[0]
print('Procesando:', video_path)

landmarks, meta = pose.extract_landmarks(video_path)

print('shape [frames, 33, 4]:', landmarks.shape)
for k, v in meta.items():
    print(f'  {k}: {v}')

In [ ]:
# 4) Dibujar el esqueleto sobre el primer frame CON pose detectada
#    (busca el primer frame no-NaN para evitar mostrar uno vacío)
valid = np.where(~np.isnan(landmarks[:, 0, 0]))[0]
assert len(valid) > 0, 'Ningun frame con pose detectada. Revisar encuadre/luz del video.'
frame_idx = int(valid[0])
print(f'Primer frame con pose: {frame_idx} (de {meta["n_frames"]})')

cap = cv2.VideoCapture(video_path)
cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
ok, frame_bgr = cap.read()
cap.release()
assert ok, 'No se pudo leer el frame.'

annotated = pose.draw_skeleton(frame_bgr, landmarks[frame_idx])

plt.figure(figsize=(10, 6))
plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.title(f'{os.path.basename(video_path)} — frame {frame_idx} — '
          f'{meta["pct_high_confidence"]:.1f}% frames buena confianza')
plt.axis('off')
plt.show()